# DGS Profile-based Models

This notebook introduces profile-based model workflows in DGS, focusing on ChromBPNet, Enformer, and Borzoi-style models. It extends scalar sequence modeling to position-aware profile and track outputs.


## Tutorial series map

This notebook is part of the `Tutorials/0_DGS_usage_models` sequence:
- `0_DGS_data.ipynb`: data, IO, intervals, targets, and loaders
- `1_DGS_models.ipynb`: binary classification and regression scalar models
- **`2_DGS_profile_models.ipynb`: profile/count and long-sequence track models**
- `3_DGS_gLMs.ipynb`: genomic language model adapters and downstream heads
- `4_DGS_explain.ipynb`: interpretation methods, attribution artifacts, and motif discovery

The notebooks are designed to be read in order, but each one keeps its examples self-contained and guarded against missing optional dependencies.


## Audience, prerequisites, and learning goals

This notebook is for users who already know basic DGS sequence modeling and want to move from scalar predictions to profile or track predictions.

Prerequisites:

- Basic PyTorch tensor shapes.
- One-hot DNA sequence encoding.
- Familiarity with DGS model imports.

By the end, you should be able to:

- Explain the difference between scalar, profile/count, and long-sequence track models.
- Instantiate lightweight DGS versions of ChromBPNet, Enformer, and Borzoi.
- Check input and output shape conventions.
- Use `ProfileCountLoss` and `calculate_profile_metrics` for profile-model workflows.
- Replace synthetic tensors with real FASTA, BED, and BigWig inputs.


## Outline

1. Set up a safe tutorial environment.
2. Inspect DGS model-zoo metadata for profile-based models.
3. Build synthetic DNA and profile tensors.
4. Run a ChromBPNet-style profile/count example.
5. Run an Enformer-style long-sequence track example.
6. Run a Borzoi-style multi-head profile example.
7. Connect the same ideas to real BigWig data and external Keras checkpoints.
8. Review exercises, pitfalls, and extensions.


## Series-level conventions

Across these notebooks:

- Core examples use tiny synthetic data or local fixtures and avoid model downloads.
- Optional real-data or external-checkpoint cells are disabled with `RUN_*` flags until you edit paths and opt in.
- Loader sequence batches are usually `(batch, length, 4)`; CNN-style models usually expect `(batch, 4, length)` after `transpose(1, 2)`.
- Scalar targets use `(batch, tasks)`; profile targets use `(batch, tracks, profile_length)`.
- Environment guards print skip messages when optional packages such as `torch`, `pysam`, `pyBigWig`, `transformers`, or `evo2` are unavailable.


## 1. Environment setup

The tutorial uses small synthetic tensors so it does not download external checkpoints or require large genomic files. If `torch` or the local DGS package is unavailable, the notebook prints a clear message and skips model execution cells.


In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "DGS").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import torch
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = repr(exc)

DGS_READY = False
if TORCH_AVAILABLE:
    try:
        from DGS.Model import ChromBPNet, Enformer, Borzoi, KerasProfileAdapter, get_model_zoo, load_keras_profile_model
        from DGS.DL.Profile import ProfileCountLoss, calculate_profile_metrics
        DGS_READY = True
    except Exception as exc:
        DGS_IMPORT_ERROR = repr(exc)

print(f"Repository root: {REPO_ROOT}")
print(f"Torch available: {TORCH_AVAILABLE}")
if not TORCH_AVAILABLE:
    print(f"Torch import error: {TORCH_IMPORT_ERROR}")
print(f"DGS model API ready: {DGS_READY}")
if TORCH_AVAILABLE and not DGS_READY:
    print(f"DGS import error: {DGS_IMPORT_ERROR}")


## 2. Profile-based models in the DGS model zoo

DGS exposes several models that produce profile-like outputs. The most important distinction is the output contract:

- `ChromBPNet`: returns `(profile_logits, log_counts)` for BPNet-style profile/count training.
- `Enformer`: returns long-sequence regulatory tracks in channel-last form, usually `(batch, target_length, tracks)`.
- `Borzoi`: returns long-sequence profile tracks in DGS-native channel-first form, usually `(batch, tracks, profile_length)`.
- `KerasProfileAdapter`: wraps external TensorFlow/Keras profile models and normalizes their outputs into DGS conventions.


In [ ]:
if DGS_READY:
    model_zoo = get_model_zoo()
    for name in ["ChromBPNet", "Enformer", "Borzoi", "KerasProfileAdapter"]:
        meta = model_zoo[name]
        print(f"{name}")
        print(f"  input_shape:  {meta['input_shape']}")
        print(f"  output_shape: {meta['output_shape']}")
        print(f"  task_type:    {meta['task_type']}")
        print(f"  status:       {meta['workflow_status']}")
        print()
else:
    print("Skipped: DGS model API is not ready in this kernel.")


## 3. Build synthetic sequence and profile tensors

Real profile workflows usually start from:

- FASTA sequence windows.
- BED intervals defining fixed-width windows.
- BigWig tracks containing profile targets.

For a fast tutorial, we create synthetic tensors directly. The key shape convention is:

- Sequence loader outputs often use channel-last `(batch, length, 4)`.
- Many convolutional models use channel-first `(batch, 4, length)`.
- DGS profile targets use channel-first `(batch, tracks, profile_length)`.


In [ ]:
if DGS_READY:
    torch.manual_seed(7)

    batch_size = 2
    sequence_length = 512
    profile_length = 64
    n_tracks = 2

    # Channel-last one-hot sequence, matching many sequence-loader outputs.
    base_indices = torch.randint(0, 4, (batch_size, sequence_length))
    seq_nlc = torch.nn.functional.one_hot(base_indices, num_classes=4).float()

    # Channel-first version for models that expect Conv1d-style DNA input.
    seq_ncl = seq_nlc.transpose(1, 2).contiguous()

    # Synthetic non-negative profile targets: (N, C, L).
    profile_targets = torch.poisson(torch.rand(batch_size, n_tracks, profile_length) * 5.0)

    print("seq_nlc:", tuple(seq_nlc.shape), "= (batch, length, 4)")
    print("seq_ncl:", tuple(seq_ncl.shape), "= (batch, 4, length)")
    print("profile_targets:", tuple(profile_targets.shape), "= (batch, tracks, profile_length)")
else:
    print("Skipped: DGS model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    def normalize_profile_per_track(profile_ncl, eps=1e-6):
        # Convert non-negative profile values to per-track probability profiles.
        return profile_ncl / profile_ncl.sum(dim=-1, keepdim=True).clamp_min(eps)

    def count_parameters(model):
        return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

    def print_tensor(name, tensor):
        print(f"{name:24s} shape={tuple(tensor.shape)} dtype={tensor.dtype}")
else:
    print("Skipped: helper functions require DGS_READY.")


## 4. ChromBPNet: profile logits plus count prediction

ChromBPNet-style models are useful when the task is base-resolution or bin-resolution profile prediction and the total signal count also matters.

The native DGS `ChromBPNet` returns:

- `profile_logits`: unnormalized profile logits with shape `(batch, tracks, profile_length)`.
- `log_counts`: log-count predictions with shape `(batch, tracks)`.

Use `ProfileCountLoss` when training with profile targets. If count targets are not passed explicitly, DGS can derive them by summing the profile targets.


In [ ]:
if DGS_READY:
    chrombpnet = ChromBPNet(
        input_len=sequence_length,
        output_len=profile_length,
        n_filters=16,
        n_layers=3,
        n_outputs=n_tracks,
        padding="same",
    )
    chrombpnet.eval()

    with torch.no_grad():
        chrom_profile_logits, chrom_log_counts = chrombpnet(seq_ncl)

    print(f"ChromBPNet trainable parameters: {count_parameters(chrombpnet):,}")
    print_tensor("profile logits", chrom_profile_logits)
    print_tensor("log counts", chrom_log_counts)
else:
    print("Skipped: DGS model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    criterion = ProfileCountLoss(
        profile_weight=1.0,
        count_weight=0.5,
        normalize_profile_by_counts=True,
        count_loss="mse",
    )

    # During training, remove torch.no_grad() and call loss.backward().
    chrom_loss = criterion((chrom_profile_logits, chrom_log_counts), profile_targets)
    print(f"Composite profile/count loss: {chrom_loss.item():.4f}")

    chrom_profile_prob = torch.softmax(chrom_profile_logits, dim=-1)
    target_profile_prob = normalize_profile_per_track(profile_targets)
    metrics = calculate_profile_metrics(target_profile_prob, chrom_profile_prob)
    display(metrics)
else:
    print("Skipped: DGS model API is not ready in this kernel.")


### Interpreting the ChromBPNet example

The profile head answers: where inside this window should signal occur?

The count head answers: how much total signal should this window have?

For real training, the target profile should come from aligned BigWig signal over fixed-width intervals. The profile length in the BigWig extraction and the model `output_len` must match.


## 5. Enformer: long-sequence regulatory track prediction

Enformer-style models use longer sequence context and usually predict many regulatory tracks across genomic bins. In DGS, the Enformer implementation uses channel-last DNA input `(batch, length, 4)` and returns channel-last tracks `(batch, target_length, tracks)` for the selected head.

The following example is intentionally tiny compared with a real Enformer model. It is only for API and shape learning.


In [ ]:
if DGS_READY:
    enformer_sequence_length = 1024
    enformer_target_length = 8
    enformer_tracks = 3

    enformer_seq_indices = torch.randint(0, 4, (batch_size, enformer_sequence_length))
    enformer_seq_nlc = torch.nn.functional.one_hot(enformer_seq_indices, num_classes=4).float()

    enformer = Enformer(
        dim=256,
        depth=1,
        heads=4,
        output_heads={"demo": enformer_tracks},
        target_length=enformer_target_length,
        dropout_rate=0.0,
        attn_dropout=0.0,
        pos_dropout=0.0,
    )
    enformer.eval()

    with torch.no_grad():
        enformer_tracks_nlc, enformer_embeddings = enformer(
            enformer_seq_nlc,
            head="demo",
            return_embeddings=True,
        )

    print(f"Enformer trainable parameters: {count_parameters(enformer):,}")
    print_tensor("input sequence", enformer_seq_nlc)
    print_tensor("predicted tracks", enformer_tracks_nlc)
    print_tensor("trunk embeddings", enformer_embeddings)
else:
    print("Skipped: DGS model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    # Example metric calculation with synthetic channel-last targets.
    enformer_targets_nlc = torch.rand_like(enformer_tracks_nlc)
    enformer_metrics = calculate_profile_metrics(enformer_targets_nlc, enformer_tracks_nlc)
    display(enformer_metrics)
else:
    print("Skipped: DGS model API is not ready in this kernel.")


### Interpreting the Enformer example

Use Enformer-style models when long sequence context is part of the scientific question. Typical use cases include distal regulatory effects, multi-track genome annotation, and extracting sequence embeddings for downstream models.

For training from profile tracks, make sure target orientation matches the model output. Enformer returns channel-last tracks, while DGS profile helper functions normalize both channel-last and channel-first small-track tensors when possible.


## 6. Borzoi: multi-head profile prediction

Borzoi-style models also predict long-range regulatory profiles, but the DGS-native module returns profile tracks in channel-first form `(batch, tracks, profile_length)`. It supports named output heads, such as human and mouse.

The example below uses a compact Borzoi-like configuration for tutorial speed.


In [ ]:
if DGS_READY:
    borzoi = Borzoi(
        input_len=sequence_length,
        output_heads={"human": n_tracks, "mouse": 1},
        target_length=profile_length,
        stem_filters=16,
        filters_init=16,
        filters_end=32,
        res_blocks=2,
        transformer_depth=1,
        transformer_heads=2,
        upsample_blocks=2,
        final_filters=32,
        dropout=0.0,
    )
    borzoi.eval()

    with torch.no_grad():
        borzoi_human_profile, borzoi_embeddings = borzoi(
            seq_ncl,
            head="human",
            return_embeddings=True,
        )
        borzoi_all_heads = borzoi(seq_ncl, head=None)

    print(f"Borzoi trainable parameters: {count_parameters(borzoi):,}")
    print_tensor("human profile", borzoi_human_profile)
    print_tensor("embeddings", borzoi_embeddings)
    print("available heads:", list(borzoi_all_heads.keys()))
    for head_name, head_output in borzoi_all_heads.items():
        print_tensor(f"head={head_name}", head_output)
else:
    print("Skipped: DGS model API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    borzoi_metrics = calculate_profile_metrics(profile_targets, borzoi_human_profile)
    display(borzoi_metrics)
else:
    print("Skipped: DGS model API is not ready in this kernel.")


### Interpreting the Borzoi example

Borzoi is a good fit when you want a profile model with named heads and long-sequence architecture components. In a production workflow, you would use the model dimensions, sequence length, target length, and head definitions that match the biological assay and checkpoint.


## 7. Replace synthetic tensors with real FASTA, BED, and BigWig data

The synthetic tensors above are useful for learning model contracts. Real profile training usually starts with `build_profile_dataloader`.

The cell below is a template. It is disabled by default because it requires local FASTA, BED, and BigWig files.


In [ ]:
RUN_REAL_BIGWIG_EXAMPLE = False

if RUN_REAL_BIGWIG_EXAMPLE:
    from DGS.Data import build_profile_dataloader

    fasta_path = "path/to/genome.fa"
    intervals_path = "path/to/fixed_width_windows.bed"
    profile_tasks = [
        {
            "task_name": "ATAC_plus",
            "file_path": "path/to/atac.plus.bw",
            "file_type": "bigwig",
            "task_type": "profile",
            "bin_size": 1,
        },
        {
            "task_name": "ATAC_minus",
            "file_path": "path/to/atac.minus.bw",
            "file_type": "bigwig",
            "task_type": "profile",
            "bin_size": 1,
        },
    ]

    loader = build_profile_dataloader(
        fasta_path=fasta_path,
        intervals_path=intervals_path,
        profile_tasks=profile_tasks,
        batch_size=4,
        return_counts=True,
        strand_aware=True,
    )

    sequences_nlc, (profiles_ncl, counts) = next(iter(loader))
    sequences_ncl = sequences_nlc.transpose(1, 2).contiguous()
    profile_logits, log_counts = chrombpnet(sequences_ncl)
    loss = criterion((profile_logits, log_counts), (profiles_ncl, counts))
    print(loss)
else:
    print("Real BigWig example is disabled. Set RUN_REAL_BIGWIG_EXAMPLE = True after editing file paths.")


## 8. External ChromBPNet or Borzoi checkpoints through KerasProfileAdapter

DGS includes native PyTorch interfaces for profile-model workflows. Some official checkpoints, especially for ChromBPNet or Borzoi-style ecosystems, are distributed as Keras `.h5` files. Use `load_keras_profile_model` when you want to run those checkpoints through a DGS-compatible PyTorch module.

This cell is also disabled by default because TensorFlow and model-specific custom objects are external runtime dependencies.


In [ ]:
RUN_EXTERNAL_KERAS_EXAMPLE = False

if RUN_EXTERNAL_KERAS_EXAMPLE:
    model = load_keras_profile_model(
        "path/to/chrombpnet_or_borzoi_model.h5",
        profile_channels=1,
        custom_objects=None,
        compile=False,
    )
    model.eval()

    with torch.no_grad():
        external_output = model(seq_nlc)

    if isinstance(external_output, tuple):
        external_profile, external_counts = external_output
        print_tensor("external profile", external_profile)
        print_tensor("external counts", external_counts)
    else:
        print_tensor("external profile", external_output)
else:
    print("External Keras example is disabled. Set RUN_EXTERNAL_KERAS_EXAMPLE = True after installing TensorFlow and editing the model path.")


## 9. Model selection guide

Use this short guide when choosing among the models:

- Choose `ChromBPNet` when your target is a localized profile and total count prediction is part of the objective.
- Choose `Enformer` when long sequence context and many output tracks are central to the task.
- Choose `Borzoi` when you need long-sequence profile prediction with named heads and profile-shaped outputs.
- Choose `KerasProfileAdapter` when you already have an official Keras checkpoint and want DGS-compatible inference.

For quick debugging, always start with a small model and synthetic tensors. Scale up only after the shape contract, loss, and dataloader are correct.


## 10. Exercise

Modify the ChromBPNet example so it predicts one profile track instead of two.

Checklist:

- Set `n_outputs=1` in the model.
- Create a target tensor with shape `(batch, 1, profile_length)`.
- Run `ProfileCountLoss` successfully.


In [ ]:
if DGS_READY:
    exercise_tracks = 1
    exercise_targets = torch.poisson(torch.rand(batch_size, exercise_tracks, profile_length) * 5.0)

    exercise_model = ChromBPNet(
        input_len=sequence_length,
        output_len=profile_length,
        n_filters=16,
        n_layers=2,
        n_outputs=exercise_tracks,
        padding="same",
    )
    exercise_model.eval()

    with torch.no_grad():
        exercise_output = exercise_model(seq_ncl)

    exercise_loss = criterion(exercise_output, exercise_targets)
    print_tensor("exercise logits", exercise_output[0])
    print_tensor("exercise counts", exercise_output[1])
    print(f"exercise loss: {exercise_loss.item():.4f}")
else:
    print("Skipped: DGS model API is not ready in this kernel.")


## Common pitfalls

Pitfall 1: mixing up sequence orientation.

- Dataloader-style sequence tensors are often `(batch, length, 4)`.
- Conv1d-style models often expect `(batch, 4, length)`.
- Use `sequences.transpose(1, 2).contiguous()` when converting channel-last DNA to channel-first DNA.

Pitfall 2: mismatched profile length.

- `ChromBPNet(output_len=...)`, `Borzoi(target_length=...)`, and the BigWig extraction length must agree.
- Fixed-width BED intervals are strongly recommended for profile workflows.

Pitfall 3: applying scalar losses to profile outputs.

- Use `ProfileCountLoss` for ChromBPNet-style profile/count outputs.
- Use profile-aware metrics after aligning tensor orientation and target length.


## Optional extensions

- Train the compact ChromBPNet example for a few gradient steps on synthetic targets to verify the loss decreases.
- Replace the synthetic tensors with a small FASTA/BED/BigWig fixture.
- Add a strand-aware paired-task example with plus and minus BigWig tracks.
- Save predictions with DGS profile writers such as NPZ, HDF5, or BigWig outputs.
